# Case Capim - Analista de Crédito e ML

### imports iniciais e leitura dos dados

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Leitura dos dados
df_contratos = pd.read_excel('Base de Contratos.xlsx')
df_parcelas = pd.read_excel('Base de Parcelas.xlsx')

df_contratos.head()

,ID_CONTRATO,VALOR_EMPRESTADO,PRAZO_MESES,SCORE_CREDITO,IDADE,ESTADO,PROFISSAO,RENDA_ESTIMADA,DATA_CONTRATACAO
0,202691,2400.0,20,850.0,45,SP,autonomous,9500.0,2025-11-14
1,144498,3900.0,24,874.0,38,PR,public_agent,3050.0,2025-01-10
2,183536,3700.0,12,672.0,55,PR,autonomous,1600.0,2025-07-29
3,194201,1200.0,10,788.0,68,BA,retired,6050.0,2025-09-27
4,190540,8800.0,24,779.0,54,SP,wage_worker,1800.0,2025-09-06


In [4]:
df_parcelas.head()

,ID_CONTRATO,NUMERO_PARCELA,VALOR_PARCELA,DATA_VENCIMENTO,DATA_PAGAMENTO
0,174323,2,184.29,2025-08-15,2025-08-16
1,174323,1,184.29,2025-07-15,2025-07-16
2,174323,8,184.29,2026-02-15,NaT
3,174323,9,184.29,2026-03-15,NaT
4,174323,10,184.29,2026-04-15,NaT


## Análise exploratória inicial

In [12]:
n_contratos = df_contratos['ID_CONTRATO'].nunique()
n_parcelas = len(df_parcelas)
n_linhas_contratos = len(df_contratos)

print(f"Total de linhas na base de contratos (df_contratos): {n_linhas_contratos}")
print(f"Contratos únicos (df_contratos): {n_contratos}")
print(f"Linhas na base de parcelas (df_parcelas): {n_parcelas}")

Total de linhas na base de contratos (df_contratos): 300
Contratos únicos (df_contratos): 300
Linhas na base de parcelas (df_parcelas): 5659


In [ ]:
print("=== Nulos em df_contratos ===")
nulos_contratos = df_contratos.isnull().sum()
pct_contratos = (nulos_contratos / len(df_contratos) * 100).round(2)
resumo_contratos = pd.DataFrame({'nulos': nulos_contratos, '%': pct_contratos})
print(resumo_contratos[resumo_contratos['nulos'] > 0].to_string() or "Sem valores nulos")

print("\n=== Nulos em df_parcelas ===")
nulos_parcelas = df_parcelas.isnull().sum()
pct_parcelas = (nulos_parcelas / len(df_parcelas) * 100).round(2)
resumo_parcelas = pd.DataFrame({'nulos': nulos_parcelas, '%': pct_parcelas})
print(resumo_parcelas[resumo_parcelas['nulos'] > 0].to_string() or "Sem valores nulos")

=== Nulos em df_contratos ===
                nulos      %
SCORE_CREDITO      24   8.00
PROFISSAO          13   4.33
RENDA_ESTIMADA    108  36.00

=== Nulos em df_parcelas ===
                nulos      %
DATA_PAGAMENTO   3315  58.58


In [ ]:
print(f"ESTADO — {df_contratos['ESTADO'].nunique()} valores únicos:")
print(sorted(df_contratos['ESTADO'].dropna().unique().tolist()))

print(f"\nPROFISSAO — {df_contratos['PROFISSAO'].nunique()} valores únicos:")
print(sorted(df_contratos['PROFISSAO'].dropna().unique().tolist()))

ESTADO — 24 valores únicos:
['AC', 'AL', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR', 'RS', 'SC', 'SE', 'SP']

PROFISSAO — 7 valores únicos:
['autonomous', 'entrepreneur', 'home_worker', 'public_agent', 'retired', 'self_employed_worker', 'wage_worker']


### Removendo horario das colunas de data (Valores apenas 00:00:00, sem informação relevante)

In [15]:
colunas_data_contratos = ['DATA_CONTRATACAO']
colunas_data_parcelas  = ['DATA_VENCIMENTO', 'DATA_PAGAMENTO']

for col in colunas_data_contratos:
    df_contratos[col] = pd.to_datetime(df_contratos[col]).dt.date

for col in colunas_data_parcelas:
    df_parcelas[col] = pd.to_datetime(df_parcelas[col]).dt.date
    
df_parcelas.head()

,ID_CONTRATO,NUMERO_PARCELA,VALOR_PARCELA,DATA_VENCIMENTO,DATA_PAGAMENTO
0,174323,2,184.29,2025-08-15,2025-08-16
1,174323,1,184.29,2025-07-15,2025-07-16
2,174323,8,184.29,2026-02-15,NaT
3,174323,9,184.29,2026-03-15,NaT
4,174323,10,184.29,2026-04-15,NaT


## Comportamento de pagamento das parcelas

Para as parcelas que não possuem data de pagamento, vamos considerar a data mais recente presente na base para comparação, que no caso é 01/04/2026, ou 2026-04-01.

In [ ]:
data_referencia = pd.to_datetime(df_parcelas['DATA_PAGAMENTO'].dropna()).max()
print(f"Data de referência (máx. DATA_PAGAMENTO): {data_referencia.date()}")

data_vencimento = pd.to_datetime(df_parcelas['DATA_VENCIMENTO'])
data_pagamento  = pd.to_datetime(df_parcelas['DATA_PAGAMENTO']).fillna(data_referencia)

df_parcelas['DIAS_ATRASO'] = (data_pagamento - data_vencimento).dt.days.clip(lower=0)

print(f"\nDIAS_ATRASO — resumo:")
print(df_parcelas['DIAS_ATRASO'].describe().round(1))

df_parcelas[['ID_CONTRATO', 'NUMERO_PARCELA', 'DATA_VENCIMENTO', 'DATA_PAGAMENTO', 'DIAS_ATRASO']].head(10)

Data de referência (máx. DATA_PAGAMENTO): 2026-04-01

DIAS_ATRASO — resumo:
count    5659.0
mean       65.3
std       211.7
min         0.0
25%         0.0
50%         0.0
75%         3.0
max      1578.0
Name: DIAS_ATRASO, dtype: float64


,ID_CONTRATO,NUMERO_PARCELA,DATA_VENCIMENTO,DATA_PAGAMENTO,DIAS_ATRASO
0,174323,2,2025-08-15,2025-08-16,1
1,174323,1,2025-07-15,2025-07-16,1
2,174323,8,2026-02-15,NaT,45
3,174323,9,2026-03-15,NaT,17
4,174323,10,2026-04-15,NaT,0
5,174323,3,2025-09-15,2025-09-18,3
6,174323,7,2026-01-15,2026-02-13,29
7,174323,6,2025-12-15,2026-01-14,30
8,174323,5,2025-11-15,2025-12-08,23
9,174323,4,2025-10-15,2025-10-18,3


## Agrupando por cliente para ver atraso máximo e média de atraso para definir critério de inadimplência

In [ ]:
atraso_por_contrato = (
    df_parcelas.groupby('ID_CONTRATO')['DIAS_ATRASO']
    .agg(ATRASO_MAX='max', ATRASO_MEDIO='mean')
    .round(1)
    .reset_index()
)

atraso_por_contrato['INADIMPLENTE_OVER30'] = (atraso_por_contrato['ATRASO_MAX'] > 30).astype(int)
atraso_por_contrato['INADIMPLENTE_OVER60'] = (atraso_por_contrato['ATRASO_MAX'] > 60).astype(int)

n_total = len(atraso_por_contrato)
taxa_over30 = atraso_por_contrato['INADIMPLENTE_OVER30'].mean() * 100
taxa_over60 = atraso_por_contrato['INADIMPLENTE_OVER60'].mean() * 100

print(f"Total de contratos: {n_total}")
print(f"\nOver30 (atraso máx > 30 dias):")
print(f"  Inadimplentes : {atraso_por_contrato['INADIMPLENTE_OVER30'].sum()} contratos")
print(f"  Taxa          : {taxa_over30:.1f}%")
print(f"\nOver60 (atraso máx > 60 dias):")
print(f"  Inadimplentes : {atraso_por_contrato['INADIMPLENTE_OVER60'].sum()} contratos")
print(f"  Taxa          : {taxa_over60:.1f}%")

atraso_por_contrato.sort_values('ATRASO_MAX', ascending=False).head(10)

Total de contratos: 300

Over30 (atraso máx > 30 dias):
  Inadimplentes : 135 contratos
  Taxa          : 45.0%

Over60 (atraso máx > 60 dias):
  Inadimplentes : 103 contratos
  Taxa          : 34.3%


,ID_CONTRATO,ATRASO_MAX,ATRASO_MEDIO,INADIMPLENTE_OVER30,INADIMPLENTE_OVER60
0,2379,1578,1395.7,1,1
1,7035,1427,1014.5,1,1
2,10306,1386,835.1,1,1
8,14429,1264,831.6,1,1
6,11504,1264,730.4,1,1
10,28248,1243,1076.3,1,1
7,12443,1243,672.7,1,1
9,26714,1233,976.8,1,1
11,32416,1203,853.0,1,1
3,10461,1151,342.2,1,1
